In [ ]:
!pip install -qU transformers optimum accelerate datasets onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.3 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings("ignore")
from datasets import load_dataset, Dataset
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
import subprocess

print("Loading and preparing Bible chapters...")
dataset = load_dataset("SzuTao/KingJamesVersionBible", split="train")
df = dataset.to_pandas()
df = df.sort_values(by=['Book ID', 'Chapter Number', 'Verse Number'])

chapters = df.groupby(['Book', 'Chapter Number'], sort=False)['Text'].apply(' '.join).reset_index()
train_df = chapters.head(100).copy()

print("Generating synthetic target summaries...")
device = "cuda" if torch.cuda.is_available() else "cpu"

teacher_id = "facebook/bart-large-cnn"
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
teacher_model = AutoModelForSeq2SeqLM.from_pretrained(teacher_id).to(device)

synthetic_summaries = []
for text in train_df['Text']:
    inputs = teacher_tokenizer(text[:3000], return_tensors="pt", max_length=1024, truncation=True).to(device)
    with torch.no_grad():
        outputs = teacher_model.generate(**inputs, max_length=60, min_length=20, do_sample=False)
    synthetic_summaries.append(teacher_tokenizer.decode(outputs[0], skip_special_tokens=True))

train_df['summary'] = synthetic_summaries
dataset_split = Dataset.from_pandas(train_df).train_test_split(test_size=0.1)

print("Preparing Student Model...")
MODEL_ID = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    formatted_texts = [
        f"Summarize this text:\n{doc[:2000]}\n\nSummary: {summ}{tokenizer.eos_token}"
        for doc, summ in zip(examples["Text"], examples["summary"])
    ]
    return tokenizer(formatted_texts, max_length=512, truncation=True)

tokenized_datasets = dataset_split.map(preprocess_function, batched=True)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gpt2_bible_model",
    eval_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

print("Starting Fine-Tuning...")
trainer.train()

trainer.save_model("./gpt2_bible_model")
tokenizer.save_pretrained("./gpt2_bible_model")

print("Exporting model to ONNX format...")
export_cmd = "optimum-cli export onnx --model ./gpt2_bible_model --task text-generation-with-past ./onnx_bible_model"
subprocess.run(export_cmd, shell=True, check=True)

tokenizer.save_pretrained("./onnx_bible_model")
print("\nPipeline Complete!")

Loading and preparing Bible chapters...


README.md:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

KJV.csv:   0%|          | 0.00/4.98M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/31102 [00:00<?, ? examples/s]

Generating synthetic target summaries...


...

Epoch,Training Loss,Validation Loss
1,No log,3.282280
2,No log,3.133395
3,No log,3.125848


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Exporting model to ONNX format...


CalledProcessError: Command 'optimum-cli export onnx --model ./gpt2_bible_model --task text-generation-with-past ./onnx_bible_model' returned non-zero exit status 2.

Above is other model and single cell test script////////////////////////////////////////

In [ ]:
!pip install -q -U "optimum[exporters,onnxruntime]" transformers datasets accelerate onnx onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 19.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incom

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
import subprocess

# 2. Load and Prepare Bible Dataset
print("Loading and preparing Bible chapters...")
dataset = load_dataset("SzuTao/KingJamesVersionBible", split="train")
df = dataset.to_pandas()
df = df.sort_values(by=['Book ID', 'Chapter Number', 'Verse Number'])

# Group verses into full chapters
chapters = df.groupby(['Book', 'Chapter Number'], sort=False)['Text'].apply(' '.join).reset_index()
train_df = chapters.head(100).copy()

# 3. Generate Target Summaries using teacher model
print("Generating synthetic target summaries...")
teacher_pipe = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    device=0 if torch.cuda.is_available() else -1,
    torch_dtype=torch.float16  # float16 is fine here because this model is only used for inference
)

synthetic_summaries = []
for text in train_df['Text']:
    prompt = f"Summarize this Bible chapter concisely in 2 sentences:\n{text[:2000]}\n\nSummary:"
    res = teacher_pipe(prompt, max_new_tokens=60, return_full_text=False, do_sample=False)
    synthetic_summaries.append(res[0]['generated_text'].strip())

train_df['summary'] = synthetic_summaries

# Convert to HF Dataset format
hf_dataset = Dataset.from_pandas(train_df)
dataset_split = hf_dataset.train_test_split(test_size=0.1)

# 4. Tokenize Dataset for the Student Model
MODEL_ID = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    formatted_texts = [
        f"Summarize this text:\n{doc[:2000]}\n\nSummary: {summ}{tokenizer.eos_token}"
        for doc, summ in zip(examples["Text"], examples["summary"])
    ]
    return tokenizer(formatted_texts, max_length=512, truncation=True)

tokenized_datasets = dataset_split.map(preprocess_function, batched=True)

# 5. Fine-Tune Model
# FIX: Load student model in FP32 so Trainer's fp16=True can maintain FP32 master weights
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./causal_bible_model",
    eval_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    fp16=True,  # Enables mixed precision with proper FP32 master weights
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

print("Starting Fine-Tuning...")
trainer.train()

# Save native PyTorch checkpoint
trainer.save_model("./causal_bible_model")
tokenizer.save_pretrained("./causal_bible_model")

Loading and preparing Bible chapters...


README.md:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

KJV.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/31102 [00:00<?, ? examples/s]

Generating synthetic target summaries...


...

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Starting Fine-Tuning...


Epoch,Training Loss,Validation Loss
1,No log,2.328561
2,No log,2.295050
3,No log,2.481194


('./causal_bible_model/tokenizer_config.json',
 './causal_bible_model/special_tokens_map.json',
 './causal_bible_model/vocab.json',
 './causal_bible_model/merges.txt',
 './causal_bible_model/added_tokens.json',
 './causal_bible_model/tokenizer.json')

In [ ]:
from transformers import AutoConfig, AutoTokenizer
from optimum.exporters.onnx import main_export
import subprocess

model_dir = "./causal_bible_model"
onnx_dir = "./onnx_bible_model"

# 1. Hard-patch the config to force standard PyTorch math (Eager Attention).
# This permanently bypasses the SDPA tracing bug that causes the TorchExportError.
print("Patching configuration to disable SDPA...")
config = AutoConfig.from_pretrained(model_dir)
config._attn_implementation = "eager"
config.save_pretrained(model_dir)

# 2. Export to ONNX using the Python API to avoid optimum-cli crash bugs
print("Exporting PyTorch model to ONNX format (this may take a minute)...")
main_export(
    model_name_or_path=model_dir,
    output=onnx_dir,
    task="text-generation-with-past"
)

# 3. Ensure the tokenizer is bundled with the ONNX files
print("Saving tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_dir)
tokenizer.save_pretrained(onnx_dir)

# 4. Zip the output
print("Zipping the ONNX model for download...")
subprocess.run(f"zip -r {onnx_dir}.zip {onnx_dir}", shell=True, check=True)

print(f"\nSuccess! You can now download {onnx_dir}.zip from the files tab.")

Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Patching configuration to disable SDPA...
Exporting PyTorch model to ONNX format (this may take a minute)...


/usr/local/lib/python3.12/dist-packages/transformers/cache_utils.py:132: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not self.is_initialized or self.keys.numel() == 0:
/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:207: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:235: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record th

Saving tokenizer...
Zipping the ONNX model for download...

Success! You can now download ./onnx_bible_model.zip from the files tab.
